# Phase 1: Basic MoE Concepts & Core Implementation
## Building Blocks of Mixtral

Now we'll implement the fundamental components of Mixture of Experts to understand how they work together.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Part 1: The Expert Network

An expert is simply a feedforward neural network. In Mixtral, each expert is a small dense layer.

In [ ]:
class Expert(nn.Module):
    """A single expert = a small feedforward neural network.
    
    Structure:
        Input → Linear(d_model → d_hidden) → ReLU → Linear(d_hidden → d_model) → Output
    """
    
    def __init__(self, d_model: int, d_hidden: int):
        """Initialize an expert.
        
        Args:
            d_model: Dimension of token embeddings
            d_hidden: Dimension of hidden layer (usually 2-4x d_model)
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)
        self.activation = nn.ReLU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through expert.
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, d_model)
            
        Returns:
            Output tensor of same shape as input
        """
        return self.fc2(self.activation(self.fc1(x)))


# Test the expert
d_model = 128
d_hidden = 256
expert = Expert(d_model, d_hidden)

# Create dummy input: batch_size=2, seq_len=4, d_model=128
x = torch.randn(2, 4, d_model)
output = expert(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Expert parameters: {sum(p.numel() for p in expert.parameters())}")

## Part 2: The Router Network

The router decides which experts should process each token. It's a learned linear layer that outputs scores for each expert.

In [ ]:
class Router(nn.Module):
    """Router network that selects which experts to activate.
    
    For each token, computes a probability distribution over experts.
    """
    
    def __init__(self, d_model: int, num_experts: int):
        """Initialize router.
        
        Args:
            d_model: Dimension of input
            num_experts: Number of experts to route between
        """
        super().__init__()
        self.num_experts = num_experts
        # Simple linear layer: d_model → num_experts
        self.router_weights = nn.Linear(d_model, num_experts)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Compute expert routing probabilities.
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, d_model)
            
        Returns:
            probs: Softmax probabilities of shape (batch_size, seq_len, num_experts)
            logits: Raw scores before softmax
        """
        logits = self.router_weights(x)  # (batch, seq, num_experts)
        probs = F.softmax(logits, dim=-1)  # Probabilities sum to 1 across experts
        return probs, logits


# Test the router
num_experts = 8
router = Router(d_model, num_experts)

probs, logits = router(x)
print(f"Router output shape: {probs.shape}")
print(f"\nRouting probabilities for first token, first batch:")
print(f"Expert probabilities: {probs[0, 0].detach().numpy()}")
print(f"Sum of probabilities: {probs[0, 0].sum().item():.6f}")  # Should be 1.0

## Part 3: Top-K Selection

From the router probabilities, we select the top-2 experts for each token.

In [ ]:
def select_top_k_experts(probs: torch.Tensor, k: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    """Select top-k experts based on routing probabilities.
    
    Args:
        probs: Router probabilities of shape (batch, seq, num_experts)
        k: Number of experts to select per token
        
    Returns:
        expert_indices: Indices of top-k experts (batch, seq, k)
        expert_weights: Weights (probabilities) of selected experts (batch, seq, k)
    """
    # Get top-k values and indices
    expert_weights, expert_indices = torch.topk(probs, k, dim=-1)
    # Re-normalize weights to sum to 1 among selected experts
    expert_weights = expert_weights / (expert_weights.sum(dim=-1, keepdim=True) + 1e-9)
    
    return expert_indices, expert_weights


# Test top-k selection
expert_indices, expert_weights = select_top_k_experts(probs, k=2)

print(f"Expert indices shape: {expert_indices.shape}")
print(f"Expert weights shape: {expert_weights.shape}")
print(f"\nFor first token (batch 0, position 0):")
print(f"  Selected experts: {expert_indices[0, 0].tolist()}")
print(f"  Their weights: {expert_weights[0, 0].detach().numpy()}")
print(f"  Sum of weights: {expert_weights[0, 0].sum().item():.6f}")

# Part 4: Expert Pool

The Expert Pool is a **collection of all 8 experts** that the router can route tokens to. Instead of creating
just one expert, we create multiple independent experts that can each specialize on different types of information.

## What is an Expert Pool?

An Expert Pool is:
- A container that holds multiple Expert networks (8 in our case)
- Each expert is independent with its own weights
- All experts have the same architecture (d_model → d_hidden → d_model)
- But each expert learns DIFFERENT weights and specializes differently
- The router decides which expert(s) each token should use

## Visual Architecture

```
Expert Pool (8 Independent Experts):
┌─────────────────────────────────────────────┐
│  Expert 0: CODE Specialist                  │
│  └─ Parameters: 65,920                      │
├─────────────────────────────────────────────┤
│  Expert 1: MATH Specialist                  │
│  └─ Parameters: 65,920 (different values!)  │
├─────────────────────────────────────────────┤
│  Expert 2: ENGLISH Specialist               │
│  └─ Parameters: 65,920 (different values!)  │
├─────────────────────────────────────────────┤
│  Expert 3: REASONING Specialist             │
│  Expert 4: LANGUAGES Specialist             │
│  Expert 5: TECHNICAL Specialist             │
│  Expert 6: CREATIVE Specialist              │
│  Expert 7: GENERAL Specialist               │
└─────────────────────────────────────────────┘
Total: 8 × 65,920 = 527,360 parameters
```

## Why We Need an Expert Pool (Instead of One Expert)

### Without Expert Pool (Bad):
```python
# Managing experts individually is messy and doesn't scale
expert_1 = Expert(128, 256)
expert_2 = Expert(128, 256)
expert_3 = Expert(128, 256)
# ... repeat 5 more times, then manually manage routing
```

### With Expert Pool (Good):
```python
# Clean, organized, scales easily
expert_pool = ExpertPool(num_experts=8, d_model=128, d_hidden=256)
```

## How Tokens Get Routed Through Expert Pool

When a token arrives, here's what happens:

```
Token: "def calculate(x):"  (Python code)
    ↓
STEP 1: Router analyzes the token
    "This looks like CODE"
    Confidence scores: [0.95, 0.05, 0.02, ...]
    ↓
STEP 2: Select top-2 experts
    Expert 0 (CODE): 95% confidence ← Selected
    Expert 7 (GENERAL): 30% confidence ← Selected
    ↓
STEP 3: Both experts process the token
    Expert 0 processes with CODE knowledge
    └─ Uses its own weights (learned during training)
    Expert 7 processes with GENERAL knowledge
    └─ Uses different weights (learned during training)
    ↓
STEP 4: Combine results
    Output = 0.75 × Expert0_output + 0.25 × Expert7_output
    ↓
Enhanced token representation
````
```

**Why this matters**:
- Same architecture = tokens can be routed flexibly to any expert
- Different weights = each expert learns specialized knowledge
- This enables specialization without routing constraints


## Specialization Through Different Data

During training, specialization emerges naturally:

```
Training Process:
1. Router sees CODE token → routes to Expert 0
2. Expert 0 updates its weights to handle CODE better
3. Loss decreases ✓

1. Router sees MATH token → routes to Expert 1
2. Expert 1 updates its weights to handle MATH better
3. Loss decreases ✓

After many iterations:
├─ Expert 0's weights optimize for CODE
├─ Expert 1's weights optimize for MATH
├─ Expert 2's weights optimize for ENGLISH
└─ etc...

Result: Each expert becomes specialized!
```

## Efficiency Gain

Without Expert Pool (Single Dense Expert):
```
Every token uses all parameters:
Token → [All 65,920 params] → Output
Result: 100% of parameters used, always
```

With Expert Pool:
```
Every token uses only 2 of 8 experts:
Token → Router → [2 experts × 65,920 params] → Output
Result: Only ~25% of parameters used per token
       + Specialization
       = Much more efficient!
```

## Real-World Example

Think of it like a helpdesk with specialists:

```
Customer calls: "My code has a bug"
    ↓
Dispatcher (Router): "This is a PROGRAMMING issue"
    ↓
Sends to: Programmer (Expert 0) + General Support (Expert 7)
    ↓
Programmer provides code fix (95% of solution)
General Support provides documentation (5% of solution)
    ↓
Customer gets excellent help because specialists were consulted!

If there was only ONE generalist:
└─ They'd know less about coding specifics
└─ Solution would be lower quality
```

In [ ]:
class ExpertPool(nn.Module):
    """Collection of all experts."""
    
    def __init__(self, num_experts: int, d_model: int, d_hidden: int):
        """Initialize pool of experts.
        
        Args:
            num_experts: Number of experts
            d_model: Dimension of embeddings
            d_hidden: Hidden dimension per expert
        """
        super().__init__()
        self.experts = nn.ModuleList([
            Expert(d_model, d_hidden) for _ in range(num_experts)
        ])
        self.num_experts = num_experts
    
    def forward(self, x: torch.Tensor, expert_idx: int) -> torch.Tensor:
        """Forward input through specific expert.
        
        Args:
            x: Input tensor
            expert_idx: Index of expert to use
            
        Returns:
            Output from selected expert
        """
        return self.experts[expert_idx](x)


# Test expert pool
expert_pool = ExpertPool(num_experts, d_model, d_hidden)

# Run through different experts
expert_0_out = expert_pool(x, 0)
expert_1_out = expert_pool(x, 1)

print(f"Expert pool created with {num_experts} experts")
print(f"Total parameters per expert: {sum(p.numel() for p in expert_pool.experts[0].parameters())}")
print(f"Total parameters in pool: {sum(p.numel() for p in expert_pool.parameters())}")
print(f"\nExpert outputs are different (as expected):")
print(f"  Expert 0 output mean: {expert_0_out.mean().item():.4f}")
print(f"  Expert 1 output mean: {expert_1_out.mean().item():.4f}")

In [ ]:
# Additional: Detailed Expert Pool Analysis

print("="*70)
print("EXPERT POOL DEEP DIVE: Parameter Distribution")
print("="*70)

# Analyze each expert
for expert_idx in range(num_experts):
    expert = expert_pool.experts[expert_idx]
    params = sum(p.numel() for p in expert.parameters())
    print(f"\nExpert {expert_idx}:")
    print(f"  Total parameters: {params:,}")
    
    # Break down by layer
    fc1_params = sum(p.numel() for p in expert.fc1.parameters())
    fc2_params = sum(p.numel() for p in expert.fc2.parameters())
    print(f"  ├─ fc1 (128→256): {fc1_params:,} parameters")
    print(f"  └─ fc2 (256→128): {fc2_params:,} parameters")

print("\n" + "="*70)
print("TOTAL POOL STATISTICS")
print("="*70)
total_params = sum(p.numel() for p in expert_pool.parameters())
print(f"Total parameters in pool: {total_params:,}")
print(f"Parameters per expert: {total_params // num_experts:,}")
print(f"Number of experts: {num_experts}")
print(f"\nMemory footprint (assuming float32):")
print(f"  Per expert: {(total_params // num_experts) * 4 / 1024:.2f} KB")
print(f"  Total pool: {total_params * 4 / 1024 / 1024:.2f} MB")

print("\n" + "="*70)
print("PARAMETER UNIQUENESS: Are Expert Weights Different?")
print("="*70)

# Compare first expert with others
expert_0_params = list(expert_pool.experts[0].parameters())
print("\nComparing Expert 0 with other experts:")
for expert_idx in range(1, min(3, num_experts)):
    expert_x_params = list(expert_pool.experts[expert_idx].parameters())
    
    # Check if weights are different
    weights_differ = not torch.allclose(expert_0_params[0], expert_x_params[0])
    
    print(f"\nExpert 0 vs Expert {expert_idx}:")
    print(f"  fc1.weight different? {weights_differ}")
    
    if weights_differ:
        # Show a sample of the differences
        diff = (expert_0_params[0] - expert_x_params[0]).abs().mean()
        print(f"  Average difference: {diff.item():.6f}")
        print(f"  Max difference: {(expert_0_params[0] - expert_x_params[0]).abs().max().item():.6f}")
    
print("\n✓ Good! Each expert has unique, learned weights.")

print("\n" + "="*70)
print("HOW EXPERTS ARE ACCESSED")
print("="*70)

print("\nDirect indexing:")
print(f"  expert_pool.experts[0] → Expert(d_model=128, d_hidden=256)")
print(f"  expert_pool.experts[1] → Expert(d_model=128, d_hidden=256)")
print(f"  expert_pool.experts[7] → Expert(d_model=128, d_hidden=256)")

print("\nUsing .forward():")
print(f"  expert_pool.forward(x, expert_idx=0)")
print(f"  expert_pool.forward(x, expert_idx=5)")

print("\nIn the MoE layer:")
print(f"  Expert selected by router: expert_idx")
print(f"  Call: expert_pool(x, expert_idx)")
print(f"  This activates ONLY that expert, not all 8!")

## Part 5: Complete MoE Layer

Now combine all components into a single MoE layer.

In [ ]:
class MoELayer(nn.Module):
    """Complete Mixture of Experts layer.
    
    For each token:
        1. Router decides which 2 experts to use
        2. Both experts process the token
        3. Results are weighted-combined using router probabilities
    """
    
    def __init__(self, d_model: int, d_hidden: int, num_experts: int, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        self.router = Router(d_model, num_experts)
        self.expert_pool = ExpertPool(num_experts, d_model, d_hidden)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, dict]:
        """Forward pass through MoE layer.
        
        Args:
            x: Input of shape (batch_size, seq_len, d_model)
            
        Returns:
            output: Processed output, same shape as input
            stats: Dictionary with routing information (for debugging/analysis)
        """
        batch_size, seq_len, d_model = x.shape
        
        # Step 1: Get router probabilities
        probs, logits = self.router(x)  # (batch, seq, num_experts)
        
        # Step 2: Select top-k experts
        expert_indices, expert_weights = select_top_k_experts(probs, self.top_k)
        # (batch, seq, top_k) each
        
        # Step 3: Process through each selected expert and combine
        output = torch.zeros_like(x)
        
        # For each of the top-k experts
        for k in range(self.top_k):
            expert_outputs = []
            
            # For each expert that might be selected
            for expert_idx in range(self.num_experts):
                # Find which tokens selected this expert
                mask = (expert_indices[:, :, k] == expert_idx)
                
                if mask.any():
                    # Process masked tokens through this expert
                    expert_out = self.expert_pool(x, expert_idx)
                    # Weight by routing probability
                    weighted = expert_out * expert_weights[:, :, k].unsqueeze(-1)
                    # Accumulate (will add up to output)
                    output = output + weighted * mask.unsqueeze(-1).float()
        
        # Collect statistics
        stats = {
            'routing_probs': probs.detach(),
            'expert_indices': expert_indices.detach(),
            'expert_weights': expert_weights.detach(),
        }
        
        return output, stats


# Test the complete MoE layer
moe_layer = MoELayer(d_model, d_hidden, num_experts, top_k=2)
moe_output, moe_stats = moe_layer(x)

print(f"MoE output shape: {moe_output.shape}")
print(f"\nRouting statistics:")
print(f"  Expert indices (first 2 positions): {moe_stats['expert_indices'][0, :2]}")
print(f"  Expert weights (first 2 positions): {moe_stats['expert_weights'][0, :2]}")

## Part 6: Expert Load Analysis

Understanding how the router distributes tokens across experts (important for load balancing).

In [ ]:
def compute_expert_load(expert_indices: torch.Tensor, num_experts: int) -> torch.Tensor:
    """Compute how many tokens were routed to each expert.
    
    Args:
        expert_indices: Indices of selected experts (batch, seq, top_k)
        num_experts: Total number of experts
        
    Returns:
        load: Number of tokens routed to each expert
    """
    batch_size, seq_len, top_k = expert_indices.shape
    total_tokens = batch_size * seq_len
    
    # Count tokens per expert
    load = torch.zeros(num_experts, device=expert_indices.device)
    for expert_idx in range(num_experts):
        load[expert_idx] = (expert_indices == expert_idx).float().sum()
    
    return load / total_tokens  # Normalize as fraction


# Analyze load
load = compute_expert_load(moe_stats['expert_indices'], num_experts)
print(f"Expert load distribution:")
for i, l in enumerate(load):
    print(f"  Expert {i}: {l.item():.2%} of tokens")
print(f"\nIdeal load per expert: {100/num_experts:.2f}%")
print(f"Load balance quality: {load.std().item():.4f} (lower is more balanced)")

## Summary:

✅ **Expert**: A feedforward neural network that specializes in certain patterns
✅ **Router**: Neural network that decides which experts to activate
✅ **Expert Pool**: Collection of multiple experts
✅ **MoE Layer**: Complete layer that routes tokens to experts and combines outputs
✅ **Load Analysis**: Tools to measure if load is balanced

These components are the foundation of Mixtral of Experts!